# C7 · Vector and raster together: landslides, districts and slope

> **Time:** about 2 hours  
> **You need:** Parts A and B, and Lessons C1 to C6, especially reading cells with `index()` (C1), plotting with `extent=` (C2), reprojecting to UTM 45N (C4), masks and areas in km² (C5) and slope from `np.gradient` (C6).  
> **By the end you can:**
> 1. read raster values at many points at once, with rasterio's `sample()` and with xarray's `.sel(method="nearest")`, and compare the points with the whole grid,
> 2. summarise a raster inside polygons with **zonal statistics**, and burn polygons into a grid with `rasterize`,
> 3. build a **distance raster** with `distance_transform_edt`, and turn a raster mask back into polygons with `shapes`.

Part B worked with points, lines and polygons; Part C so far worked with grids. Real questions need both at once. *How steep is the ground where landslides happened? What is the mean slope of each district? How far is each place from a district headquarters?* This lesson moves values from rasters to vectors and back again.

The landslide points used here are **made-up practice data** (see `data/README.md`): they were placed at random, more often on steep slopes. They are not observations.

**Counting lines:** as in Part B, lines that only set up a plot (`fig, ax = plt.subplots()` and `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**About the numbers:** reprojection depends on the version of GDAL inside rasterio. With another version, slopes can differ from the text by a few tenths of a degree, and counts of small patches (step 8) by a few percent. The story stays the same.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 30 of 36 · [Course home](../README.md)  
← Previous: [C6 · Terrain analysis: slope, aspect and hillshade for landslide studies](C06_Terrain_analysis.ipynb) · Next: [C8 · Interpolation: a rainfall map from 24 gauges](C08_Interpolation.ipynb) →

## 1. Getting ready: one grid in metres, one CRS for everything

Raster and vector layers can only work together when they share a **CRS**, the rule you learned for vectors in B4. In this lesson everything lives on the **UTM 45N grid** (EPSG:32645), because slope and distance need metres.

The first cell rebuilds the slope grid exactly **as in C6**: reproject the DEM to UTM 45N with 30 m cells (bilinear resampling, C4), then take `np.gradient` with the real cell size of 30 m. `masked=True` turns the DEM's nodata value into `NaN` when the file is read.

`dem_utm.copy(data=slope_deg)` makes a new DataArray with the same grid, coordinates and CRS as `dem_utm`, but holding the slope values. You will use this trick several times.

The landslide points come from a CSV with `lon` and `lat`, so they start in EPSG:4326 and are converted with `to_crs(32645)`, as in B4.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rioxarray as rxr
import xarray as xr
from rasterio.enums import Resampling

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

# As in C6: reproject to UTM 45N with 30 m cells, then np.gradient with the 30 m cell size
dem = rxr.open_rasterio(DATA / "dem_kathmandu_melamchi.tif", masked=True).squeeze("band", drop=True)
dem_utm = dem.rio.reproject("EPSG:32645", resolution=30, resampling=Resampling.bilinear)

cell_m = 30
cell_km2 = cell_m * cell_m / 1000000                 # one cell is 0.0009 km²
dz_drow, dz_dcol = np.gradient(dem_utm.values, cell_m, cell_m)
slope_deg = np.degrees(np.arctan(np.sqrt(dz_drow ** 2 + dz_dcol ** 2)))
slope = dem_utm.copy(data=slope_deg)                 # same grid and CRS, slope values
valid = np.isfinite(dem_utm.values)                  # True where the DEM has data

print(slope.shape, slope.rio.crs, slope.rio.resolution())
print(round(float(np.nanmean(slope_deg)), 1), "degrees mean slope")

In [ ]:
slides = pd.read_csv(DATA / "practice_landslides_melamchi.csv")     # made-up practice data
landslides = gpd.GeoDataFrame(slides, geometry=gpd.points_from_xy(slides["lon"], slides["lat"]), crs="EPSG:4326")
landslides_utm = landslides.to_crs(32645)
print(len(landslides_utm), landslides_utm.crs.to_epsg(), slope.rio.crs.to_epsg())

left, bottom, right, top = slope.rio.bounds()
fig, ax = plt.subplots(figsize=(8, 7))   # fig: the whole figure; ax: the map area inside it (A10)
im = ax.imshow(slope_deg, extent=[left, right, bottom, top], cmap="YlOrRd", vmin=0, vmax=60)
landslides_utm.plot(ax=ax, color="royalblue", edgecolor="white", markersize=30)
fig.colorbar(im, ax=ax, label="Slope (degrees)", shrink=0.7)
ax.set_title("Slope and practice landslides (made-up points)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

The UTM grid has 1,687 rows and 1,991 columns of 30 m cells, and the mean slope is about 23.4°. Both layers report EPSG code 32645, so they can be combined.

The points sit on the orange and red ground, away from the pale valley floors. Step 4 measures that impression. Thin slivers along the edges have no data (`NaN`): the grid in degrees became a slightly tilted shape when it was reprojected (C4).

The `1e6` above the y axis means "times one million": the tick 3.07 is a northing of 3,070,000 m.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). The full licence notice is in `data/README.md`.*

### Practice

**Q1 · one line:** What is the steepest slope in the grid? Use `np.nanmax` on `slope_deg`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.nanmax(slope_deg)
```

About 73.7°. `np.nanmax` skips the `NaN` cells; plain `np.max` would return `nan`, as you saw in C5.
</details>

**Q2 · two lines:** Read `district_headquarters.geojson`, convert it to UTM 45N, and print the names of the headquarters inside the grid. Use `.cx[left:right, bottom:top]` from B5 with the bounds from the example.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hq_check = gpd.read_file(DATA / "district_headquarters.geojson").to_crs(32645)
print(hq_check.cx[left:right, bottom:top]["name"].tolist())
```

Five headquarters lie inside: Bhaktapur, Chautara, Dhulikhel, Kathmandu and Lalitpur. `.cx` works on the UTM layer because `left`, `right`, `bottom` and `top` are UTM metres too.
</details>

**Q3 · three lines:** How much of the area with data is steeper than 30°? Count the cells of `slope_deg` above 30, turn the count into km² with `cell_km2`, and print it next to the total area with data (`valid.sum()` cells).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
steep_km2 = (slope_deg > 30).sum() * cell_km2
total_km2 = valid.sum() * cell_km2
print(round(steep_km2, 1), round(total_km2, 1))
```

About 886.7 km² of 2,948.4 km² is steeper than 30°, roughly 30%. `NaN > 30` is `False`, so the empty corners are not counted.
</details>

## 2. Values at points with rasterio's `sample()`

In C1 you found the elevation at Kathmandu Durbar Square with `src.index()` and then looked up one cell of the array. For many points, rasterio offers `src.sample(coords)`.

- `coords` is a list of `(x, y)` pairs **in the raster's CRS**. The DEM file is in EPSG:4326, so the pairs are `(lon, lat)`: x first, as always.
- `sample()` reads only the cells it needs, so it stays fast even for thousands of points.
- It returns a **generator**: a source that hands out one result at a time while you loop over it. Each result is a small array with one value per band.

`[v[0] for v in src.sample(coords)]` loops over the results and keeps band 1 of each, in a list comprehension (A3).

Neither rasterio nor xarray converts coordinates for you; the second cell shows what happens if you forget.

**Predict:** in C1, about 15.8% of the DEM's cells lay above 2,000 m. Will the share of practice landslides above 2,000 m be higher or lower?

In [ ]:
coords = list(zip(slides["lon"], slides["lat"]))     # (x, y) = (lon, lat): the file's own CRS
print(coords[0])

with rasterio.open(DATA / "dem_kathmandu_melamchi.tif") as src:
    print(src.crs)
    values = [v[0] for v in src.sample(coords)]

slides["elev_m"] = values
print(slides[["id", "lon", "lat", "elev_m"]].head())
print(slides["elev_m"].describe().round(1))

In [ ]:
# THE MISTAKE, ON PURPOSE: UTM metres given to a file in degrees
x0, y0 = landslides_utm.geometry.x.iloc[0], landslides_utm.geometry.y.iloc[0]
print(round(x0), round(y0))
with rasterio.open(DATA / "dem_kathmandu_melamchi.tif") as src:
    print([int(v[0]) for v in src.sample([(x0, y0)])])

Landslide M001 lies at 1,234 m. The 60 points range from 745 m to 3,692 m, with a median of about 1,604 m. Q3 below answers the Predict question.

`zip` pairs each longitude with its latitude, as in B7. The values come straight from the original file in degrees, with no resampling: they are whole metres, like the DEM itself (`int16`).

In the second cell, M001's easting and northing (365,183 m and 3,063,795 m) were read as if they were degrees. That spot lies far outside the DEM, so `sample()` returns −32768, the **nodata** value (C1), without an error. Some versions also print a `RuntimeWarning` that does not say what went wrong. A value of −32768, or `NaN`, at a point is a signal: check that the points and the raster share a CRS.

### Practice

**Q1 · one line:** Show the `id` and `elev_m` of the highest landslide. Sort `slides` by `elev_m` and take the last row.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
slides.sort_values("elev_m")[["id", "elev_m"]].tail(1)
```

M046 is highest, at 3,692 m. `sort_values` works on the new column like on any other column (A9).
</details>

**Q2 · two lines:** Check `sample()` against C1: sample the DEM at Kathmandu Durbar Square, `(85.307, 27.704)`, and print the value. Put the one pair inside a list, because `sample()` expects a list of pairs.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(DATA / "dem_kathmandu_melamchi.tif") as src:
    print(list(src.sample([(85.307, 27.704)]))[0][0])
```

It prints 1,300 m, the same cell that `src.index(85.307, 27.704)` found in C1 (row 1066, column 385). `list(...)` collects the generator's single result; `[0][0]` takes the first point, then its first band.
</details>

**Q3 · three lines:** How many landslides lie above 2,000 m, and what percentage of all 60 is that? Filter `slides`, then print the count and the percentage rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
high = slides[slides["elev_m"] > 2000]
print(len(high))
print(round(len(high) / len(slides) * 100, 1))
```

15 landslides, 25.0% of the records, lie above 2,000 m: a higher share than the 15.8% of all cells in C1. Once the values are in a column, the whole of A9 applies.
</details>

## 3. Values at points with xarray's `.sel(method="nearest")`

The slope grid exists only in memory, as a DataArray, so `sample()` cannot read it. xarray has its own tool: `slope.sel(x=..., y=..., method="nearest")` picks the cell whose centre is **nearest** to each coordinate. The coordinates must be in the DataArray's CRS, here UTM metres.

For one point, give two numbers. For many points, wrap the x values and the y values in `xr.DataArray(..., dims="point")`. The shared name `"point"` tells xarray to **pair** them: first x with first y, second x with second y. This is called **pointwise** (or vectorised) selection.

Without that wrapper, xarray takes every x with every y. The second cell shows that mistake on purpose.

**Predict:** the second cell passes 60 x values and 60 y values as plain arrays. What shape will the result have?

In [ ]:
x_m = xr.DataArray(landslides_utm.geometry.x.values, dims="point")
y_m = xr.DataArray(landslides_utm.geometry.y.values, dims="point")

at_points = slope.sel(x=x_m, y=y_m, method="nearest")
print(at_points.shape)

slides["slope_deg"] = at_points.values.astype(float)   # float32 to float (A8): prints neatly
print(slides[["id", "elev_m", "slope_deg"]].head())
print(slides["slope_deg"].describe().round(1))

In [ ]:
# THE MISTAKE, ON PURPOSE: plain arrays, not paired
block = slope.sel(x=landslides_utm.geometry.x.values, y=landslides_utm.geometry.y.values, method="nearest")
print(block.shape)

With paired coordinates you get 60 values, one per landslide. M001 sits on a slope of about 52.0°. The landslide slopes range from about 23.0° to 59.9°, with a median of about 36.6°.

Without pairing, the shape is `(60, 60)`: 3,600 values, most of them at places where no landslide is. No error appears, so check `.shape` every time.

### Practice

**Q1 · one line:** Read the slope at the first landslide only. Give `.sel` two plain numbers: `landslides_utm.geometry.x.iloc[0]` and `landslides_utm.geometry.y.iloc[0]`, and wrap the result in `float()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
float(slope.sel(x=landslides_utm.geometry.x.iloc[0], y=landslides_utm.geometry.y.iloc[0], method="nearest"))
```

About 52.0°, the same as the first value of the pointwise selection. `float()` turns the one-cell DataArray into a plain number.
</details>

**Q2 · two lines:** Read the elevation at the landslides from `dem_utm` with `x_m` and `y_m`, store it in a column `elev_utm_m`, and print the mean absolute difference from `elev_m` (use `.abs().mean()`).

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
slides["elev_utm_m"] = dem_utm.sel(x=x_m, y=y_m, method="nearest").values
print((slides["elev_utm_m"] - slides["elev_m"]).abs().mean())
```

The two differ by about 5.5 m on average. The UTM grid was resampled with bilinear interpolation (C4), and its cell centres are not the original ones. Sample the original file when exact source values matter.
</details>

**Q3 · three lines:** Which landslides lie on slopes steeper than 45°? Filter `slides`, then print how many there are and their `id`, `slope_deg` and `area_m2`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
very_steep = slides[slides["slope_deg"] > 45]
print(len(very_steep))
print(very_steep[["id", "slope_deg", "area_m2"]].round(1))
```

7 of the 60 practice landslides sit on slopes steeper than 45°; the steepest is M050 at about 59.9°.
</details>

## 4. Points against the whole grid

A mean slope of 36° at the landslides means little on its own. The question is: **compared with what?** The fair comparison is the background, every valid cell of the grid.

`slope_deg[np.isfinite(slope_deg)]` keeps only the valid cells, as one long 1-D array (C5). Then compare medians, and the share of cells and of landslides above a few thresholds.

For a picture, draw both as histograms with `density=True`. Each histogram is then scaled so that its total area is 1, and 60 points and 3 million cells fit on one axis.

The points are made-up, and were placed more often on steep ground, so a difference is expected. With real records, this comparison is the first step of a landslide **susceptibility** study: which ground is more prone to failure.

In [ ]:
all_cells = slope_deg[np.isfinite(slope_deg)]        # every valid cell, as a 1-D array
print(len(all_cells), "cells")
print("median slope, all cells:", round(float(np.median(all_cells)), 1))
print("median slope, landslides:", round(slides["slope_deg"].median(), 1))

for limit in [15, 30, 45]:
    share_cells = (all_cells > limit).mean() * 100
    share_slides = (slides["slope_deg"] > limit).mean() * 100
    print(f"steeper than {limit}°: {share_cells:.1f}% of cells, {share_slides:.1f}% of landslides")

In [ ]:
bins = np.arange(0, 80, 5)            # 0, 5, 10, ... 75 degrees
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(all_cells, bins=bins, density=True, color="lightgrey", label="All cells")
ax.hist(slides["slope_deg"], bins=bins, density=True, histtype="step", linewidth=2,
        color="crimson", label="Practice landslides (made-up)")
ax.set_xlabel("Slope (degrees)")
ax.set_ylabel("Density (share per degree)")
ax.legend(loc="upper left")
plt.show()

The median slope of all 3,268,915 valid cells is about 24.1°; at the landslides it is about 36.6°. About 30.1% of the ground is steeper than 30°, but 76.7% of the practice landslides are. Every landslide (100%) lies on ground steeper than 15°, against 76.2% of the cells.

The red outline sits to the right of the grey bars: the landslides favour steep ground. `histtype="step"` draws only the outline, so the grey bars stay visible beneath it.

### Practice

**Q1 · one line:** Print the **mean** slope of all cells and of the landslides, each rounded to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(float(all_cells.mean()), 1), round(slides["slope_deg"].mean(), 1))
```

About 23.4° for all cells and 36.5° for the landslides. Means and medians tell the same story here.
</details>

**Q2 · two lines:** A **frequency ratio** divides the share of landslides in a class by the share of ground in that class. Work it out for slopes steeper than 30°.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
share_ground = (all_cells > 30).mean()
print(round((slides["slope_deg"] > 30).mean() / share_ground, 2))
```

About 2.54: on these made-up data, ground steeper than 30° holds two and a half times more landslides than its area alone would suggest. A ratio above 1 marks a class that favours landslides.
</details>

**Q3 · three lines:** Work out the frequency ratio for three slope classes: 15–30°, 30–45° and 45–90°. Loop over pairs of limits with `zip`, and combine two conditions with `&` (A9).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for low, high in zip([15, 30, 45], [30, 45, 90]):
    ground = ((all_cells > low) & (all_cells <= high)).mean()
    found = ((slides["slope_deg"] > low) & (slides["slope_deg"] <= high)).mean()
    print(low, high, round(found / ground, 2))
```

About 0.51 for 15–30°, 2.36 for 30–45° and 4.53 for 45–90°. The ratio rises with slope. The steepest class has few cells and only 7 points, so its ratio rests on very little data.
</details>

## 5. Zonal statistics: a raster summarised inside polygons

**Zonal statistics** summarise the raster cells inside each polygon, called a **zone**: their count, mean, maximum and so on.

`zonal_stats(polygons, array, affine=transform, nodata=..., stats=[...])` from the `rasterstats` library does this. `affine` is the grid's transform (C1), which tells it where each cell lies. It returns a **list of dictionaries**, one per polygon (A4), which `pd.DataFrame` turns into a table (A9).

A cell belongs to a zone when its **centre** lies inside the polygon, as with `all_touched=False` in C3. `nodata=np.nan` tells it to skip the `NaN` cells.

Only the part of each district covered by the DEM is summarised. For districts that are only partly inside, the numbers describe **that part only**, so report the area covered alongside them.

In [ ]:
from rasterstats import zonal_stats

districts = gpd.read_file(DATA / "nepal_districts.geojson")
near = districts.cx[85.2:85.8, 27.55:28.0].to_crs(32645).reset_index(drop=True)
print(near["district"].tolist())

stats = zonal_stats(near, slope_deg, affine=slope.rio.transform(), nodata=np.nan,
                    stats=["count", "mean", "max"])
print(stats[0])

In [ ]:
slope_table = pd.DataFrame(stats)
slope_table["district"] = near["district"]
slope_table["km2_covered"] = slope_table["count"] * cell_km2
print(slope_table.sort_values("mean", ascending=False).round(1))

`.cx` selects the 9 districts that touch the DEM's box (B5), and `reset_index(drop=True)` numbers them 0 to 8. The first dictionary, for Bhaktapur, holds 136,228 cells with a mean slope of about 11.6° and a maximum of about 48.7°.

Ranked by mean slope, Rasuwa, Makawanpur and Sindhupalchok come first (about 29.0°, 28.9° and 27.6°); the Kathmandu Valley districts, Kathmandu, Lalitpur and Bhaktapur, are the gentlest. Rasuwa's figure rests on only about 32.7 km² of the district, its southern tip. Sindhupalchok's rests on about 989.6 km².

### Practice

**Q1 · one line:** Run `zonal_stats` on the **elevation** grid `dem_utm.values` (with `dem_utm.rio.transform()`) and ask for the `"mean"` only. Show the result.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
zonal_stats(near, dem_utm.values, affine=dem_utm.rio.transform(), nodata=np.nan, stats=["mean"])
```

Nine dictionaries, one per district, in the order of `near`. Bhaktapur's mean elevation is about 1,446 m and Sindhupalchok's covered part about 1,684 m.
</details>

**Q2 · two lines:** Ask `zonal_stats` for the `"median"` and the `"percentile_90"` of slope and make a table of the result. Print it with the district names: `table.assign(district=near["district"])` adds a column and returns the new table in one go.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
p90 = pd.DataFrame(zonal_stats(near, slope_deg, affine=slope.rio.transform(), nodata=np.nan, stats=["median", "percentile_90"]))
print(p90.assign(district=near["district"]).round(1))
```

`percentile_90` is the slope that 90% of the district's cells stay below. In Sindhupalchok it is about 40.4°. `.assign` leaves `p90` itself unchanged.
</details>

**Q3 · three lines:** How much of each district does the DEM cover? Add the full district area in km² (from `near.area`, B4) to `slope_table`, then the percentage covered, and print `district` and that percentage.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
slope_table["km2_district"] = near.area / 1000000
slope_table["pct_covered"] = slope_table["km2_covered"] / slope_table["km2_district"] * 100
print(slope_table[["district", "pct_covered"]].round(0))
```

Bhaktapur is covered 100%, Kathmandu 99%, Nuwakot about 47%, Kabhrepalanchok 42%, Lalitpur 41% and Sindhupalchok 40%; Dhading, Makawanpur and Rasuwa only 1–2%. These shares agree with the coverage table of C3 to within about one percentage point; C3 counted cells on the grid in degrees, with an approximate cell area. Report these numbers with any zonal statistic for a partly covered district.
</details>

## 6. Burning polygons into a grid with `rasterize`

`rasterio.features.rasterize` turns shapes into cells. It is the vector-to-raster direction of this lesson. You give it:

- a list of `(geometry, value)` pairs: each polygon and the number to burn into its cells,
- `out_shape` and `transform`: the reference grid, here the slope grid, so the result lines up cell for cell,
- `fill`: the value for cells that no shape covers,
- `dtype`: the data type of the result. It must hold every value **and** the fill. Codes 1 to 9 and a fill of 0 fit in `uint8` (0 to 255, A8).

A cell is burned when its centre is inside the polygon; `all_touched=True` burns every cell the polygon touches (C3). Once you have a grid of zone codes, any raster on the same grid can be summarised with a NumPy mask, such as `slope_deg[zones == 4]`.

In [ ]:
from rasterio import features

near["zone"] = np.arange(1, len(near) + 1)            # one code per district: 1, 2, ... 9
pairs = list(zip(near.geometry, near["zone"]))

zones = features.rasterize(pairs, out_shape=slope.shape, transform=slope.rio.transform(),
                           fill=0, dtype="uint8")
print(zones.dtype, zones.shape)
codes, counts = np.unique(zones, return_counts=True)
print(codes)
print(counts)

In [ ]:
kathmandu_code = near.loc[near["district"] == "Kathmandu", "zone"].iloc[0]
in_kathmandu = (zones == kathmandu_code) & valid
print(kathmandu_code, round(in_kathmandu.sum() * cell_km2, 1), "km² with data")
print(round(float(np.nanmean(slope_deg[in_kathmandu])), 1), "degrees mean slope")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(zones, extent=[left, right, bottom, top], cmap="Set1", vmin=0.5, vmax=9.5, interpolation="nearest")
near.boundary.plot(ax=ax, color="black", linewidth=0.6)
ax.set_xlim(left, right)
ax.set_ylim(bottom, top)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
cbar = fig.colorbar(im, ax=ax, ticks=near["zone"], label="Zone code and district", shrink=0.7)
cbar.ax.set_yticklabels([f"{code} {name}" for code, name in zip(near["zone"], near["district"])])
ax.set_title("Districts burned into the 30 m grid")
plt.show()

`zones` is a `uint8` grid with the same shape as `slope`. Every cell got a code from 1 to 9: there is no 0, because these districts cover the whole rectangle, even its empty corners. That is why the example combines `zones == kathmandu_code` with `valid`.

Kathmandu is zone 4, with about 411.6 km² of cells with data and a mean slope of about 15.7°, the same as `zonal_stats` gave in step 5. The map shows each district as one colour, and the vector boundaries fall exactly on the colour edges. `"Set1"` has exactly nine distinct colours, one per code, and the colour bar labels each with its district. `interpolation="nearest"` keeps the codes crisp, and `set_xlim`/`set_ylim` stop the boundaries, which run far beyond the grid, from widening the map.

### Practice

**Q1 · one line:** Which district is zone 1, and how many cells does it have? Print `near.loc[0, "district"]` and the number of cells where `zones == 1`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(near.loc[0, "district"], (zones == 1).sum())
```

Bhaktapur, with 136,228 cells, exactly the count from `zonal_stats`. Bhaktapur lies wholly inside the DEM, so both methods see the same cells.
</details>

**Q2 · two lines:** Rasterize the same pairs again with `all_touched=True`, and count how many cells differ from `zones`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
zones_touched = features.rasterize(pairs, out_shape=slope.shape, transform=slope.rio.transform(), fill=0, dtype="uint8", all_touched=True)
print((zones_touched != zones).sum())
```

About 5,443 cells change, all along district borders. With `all_touched=True`, a border cell touched by two districts goes to the one burned **last**, so the order of the pairs starts to matter.
</details>

**Q3 · three lines:** Using `zones`, `valid` and `dem_utm`, print the mean elevation and the highest elevation of the part of Sindhupalchok inside the DEM.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
sindhu_code = near.loc[near["district"] == "Sindhupalchok", "zone"].iloc[0]
in_sindhu = (zones == sindhu_code) & valid
print(round(float(dem_utm.values[in_sindhu].mean()), 1), round(float(dem_utm.values[in_sindhu].max()), 1))
```

About 1,684.5 m on average and 3,977.2 m at most: the highest ground of the whole DEM is in Sindhupalchok. The values match `zonal_stats`, because both use cell centres.
</details>

## 7. Distance rasters with `distance_transform_edt`

A **distance raster** stores, in every cell, the distance to the nearest feature, such as a road, a river or a headquarters.

`scipy.ndimage.distance_transform_edt` makes one. EDT stands for **Euclidean distance transform**: straight-line distance. It works on a True/False grid and gives each `True` cell its distance to the nearest `False` cell. So mark the features as `False` and everything else as `True`.

The distance comes in cells, unless you pass `sampling=30`: then it is in metres. This only makes sense on a projected grid with square cells, such as our UTM grid.

`rasterio.transform.rowcol(transform, xs, ys)` turns map coordinates into row and column numbers (C1), so you can set the headquarters' cells to `False`. In B8, `sjoin_nearest` gave exact distances for a few points; a distance raster answers the same question for **every** cell at once.

In [ ]:
from scipy import ndimage

hq = gpd.read_file(DATA / "district_headquarters.geojson").to_crs(32645)
hq_in = hq.cx[left:right, bottom:top]
print(hq_in["name"].tolist())

rows, cols = rasterio.transform.rowcol(slope.rio.transform(), hq_in.geometry.x, hq_in.geometry.y)
not_hq = np.ones(slope.shape, dtype=bool)      # True everywhere ...
not_hq[rows, cols] = False                     # ... except the 5 headquarters cells

dist_km = ndimage.distance_transform_edt(not_hq, sampling=cell_m) / 1000
print(round(float(dist_km[valid].max()), 1), "km at most,", round(float(dist_km[valid].mean()), 1), "km on average")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(np.where(valid, dist_km, np.nan), extent=[left, right, bottom, top], cmap="viridis_r")
hq_in.plot(ax=ax, color="red", marker="^", markersize=60)
for name, point in zip(hq_in["name"], hq_in.geometry):
    ax.annotate(name, (point.x, point.y), xytext=(4, 4), textcoords="offset points", color="white")
fig.colorbar(im, ax=ax, label="Distance to nearest headquarters (km)", shrink=0.7)
ax.set_title("Straight-line distance to the 5 headquarters inside the grid")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

In [ ]:
dist = slope.copy(data=dist_km)               # a DataArray, so .sel works as in step 3
slides["hq_km"] = dist.sel(x=x_m, y=y_m, method="nearest").values
print(slides["hq_km"].describe().round(1))

Inside the area with data, no cell is more than about 34.6 km from one of the five headquarters, and the average cell is about 13.7 km away. The map shows rings around each headquarters, merging where two are close, as in the Kathmandu Valley. `np.where(valid, dist_km, np.nan)` blanks the empty corners, as in C5.

The landslides lie between about 1.4 and 33.0 km from a headquarters, 17.7 km on average. **Edge effect:** the raster only knows the features inside the grid. A headquarters a little outside, such as Dhunche or Bidur, is invisible to it, so distances near the edges can be too large. Review 2 measures this.

### Practice

**Q1 · one line:** What percentage of the area with data lies within 10 km of a headquarters? Combine `dist_km <= 10` with `valid`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round((dist_km[valid] <= 10).mean() * 100, 1))
```

About 37.5% of the area. `dist_km[valid]` keeps only cells with data, and the mean of a True/False array is the share of `True` values.
</details>

**Q2 · two lines:** Check the raster against B8: use `gpd.sjoin_nearest(landslides_utm, hq_in, distance_col="hq_m")` and print the largest difference, in km, from `slides["hq_km"]`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
exact = gpd.sjoin_nearest(landslides_utm, hq_in, distance_col="hq_m")
print((exact["hq_m"] / 1000 - slides["hq_km"]).abs().max())
```

The largest difference is about 0.02 km, less than one 30 m cell: the raster measures from cell centres, and each point is moved to its nearest centre. Both tables carry the same row labels, 0 to 59, because `landslides_utm` was built from `slides`; pandas lines the two columns up by label before subtracting.
</details>

**Q3 · three lines:** Which landslide is farthest from a headquarters? Sort `slides` by `hq_km` and print the `id`, `elev_m`, `slope_deg` and `hq_km` of the last two rows.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
farthest = slides.sort_values("hq_km")
print(farthest[["id", "elev_m", "slope_deg", "hq_km"]].tail(2).round(1))
```

M054 and M060 are farthest, about 32.6 and 33.0 km from the nearest headquarters in the grid. Review 2 checks whether a headquarters outside the grid is in fact closer.
</details>

## 8. From raster back to vector with `shapes`

`rasterio.features.shapes` does the reverse of `rasterize`: it **polygonizes** a grid. It walks over the cells and returns one polygon for each connected patch of equal value.

- Its input must be a number grid such as `uint8`; a True/False mask is not accepted, so convert it with `.astype("uint8")`.
- `mask=steep` keeps only the patches where the mask is `True`.
- It yields `(geometry, value)` pairs. Each geometry is a GeoJSON-like dictionary; `shapely.geometry.shape()` turns it into a shapely polygon (B2).
- By default, cells join only through their **edges** (`connectivity=4`); `connectivity=8` also joins cells that touch at a corner.

Below, ground steeper than 35° becomes a polygon layer, which you can buffer, join and save like any vector layer from Part B.

In [ ]:
from shapely.geometry import shape

steep = slope_deg > 35                      # True/False grid; NaN > 35 is False
print(round(steep.sum() * cell_km2, 1), "km² steeper than 35 degrees")

patches = features.shapes(steep.astype("uint8"), mask=steep, transform=slope.rio.transform())
steep_gdf = gpd.GeoDataFrame(geometry=[shape(geom) for geom, value in patches], crs=32645)
steep_gdf["area_km2"] = steep_gdf.area / 1000000
print(len(steep_gdf), "polygons,", round(steep_gdf["area_km2"].sum(), 1), "km² in total")
print(steep_gdf["area_km2"].describe().round(4))

In [ ]:
from matplotlib.colors import LightSource

big = steep_gdf[steep_gdf["area_km2"] >= 1]
print(len(big), "patches of 1 km² or more,", round(big["area_km2"].sum(), 1), "km²")

# Hillshade with the sun in the north-west (315°), 45° above the horizon, 30 m cells
shade = LightSource(azdeg=315, altdeg=45).hillshade(dem_utm.values, dx=cell_m, dy=cell_m)
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(shade, extent=[left, right, bottom, top], cmap="gray")
big.plot(ax=ax, color="crimson", edgecolor="darkred", linewidth=0.3)
ax.set_title("Patches steeper than 35°, 1 km² or larger (red)")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
plt.show()

About 479.6 km² is steeper than 35°. Polygonizing gives 22,394 polygons whose areas add up to exactly that total, a good check. Most are tiny: the median patch is only 3 cells (0.0027 km²), and the smallest are single cells of 0.0009 km².

Only 49 patches reach 1 km², but together they hold about 146.7 km². Most lie in the north and north-east, on the flanks of the deep valleys there.

The grey background is a hillshade, as in C6, so the red patches can be read against the shape of the land. This time it comes from Matplotlib's `LightSource`, which takes the sun's direction (`azdeg`, `altdeg`) and the cell size in metres (`dx`, `dy`). With some library versions, `features.shapes` prints a `DeprecationWarning` about a "Memory" driver; it is harmless.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA). The full licence notice is in `data/README.md`.*

### Practice

**Q1 · one line:** How many of the polygons are single cells? Count the rows of `steep_gdf` with an area below 0.001 km².

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
(steep_gdf["area_km2"] < 0.001).sum()
```

7,949 polygons, about a third of them, are single 30 m cells. Such specks are typical of polygonized rasters and are usually filtered out by area.
</details>

**Q2 · two lines:** Polygonize `steep` again with `connectivity=8` and print the number of polygons.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
patches_8 = features.shapes(steep.astype("uint8"), mask=steep, transform=slope.rio.transform(), connectivity=8)
print(len(list(patches_8)))
```

16,457 polygons instead of 22,394: cells that touch only at a corner now join the same patch. The total area stays the same; only how it is split changes. `list(...)` collects the generator so `len` can count it.
</details>

**Q3 · three lines:** How many practice landslides fall inside a steep patch of 0.1 km² or more? Filter `steep_gdf`, join `landslides_utm` to it with `gpd.sjoin` (B9, `predicate="within"`, inner), and count the unique `id`s.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
patches_01 = steep_gdf[steep_gdf["area_km2"] >= 0.1]
inside = gpd.sjoin(landslides_utm, patches_01, how="inner", predicate="within")
print(len(patches_01), inside["id"].nunique())
```

690 patches are 0.1 km² or larger, and 26 of the 60 made-up landslides fall inside one of them. Raster results, once polygonized, work with every vector tool from Part B.
</details>

## 9. When it breaks: a fill value that does not fit the dtype

You want the background of the zone grid to be `-5`, to make it stand out, and keep `dtype="uint8"`. Run the cell.

In [ ]:
zones_bad = features.rasterize(pairs, out_shape=slope.shape, transform=slope.rio.transform(),
                               fill=-5, dtype="uint8")

Read the last line: `OverflowError: Python integer -5 out of bounds for uint8`.

- `uint8` holds whole numbers from 0 to 255 only (A8). It has no room for a minus sign.
- **Overflow** means a value does not fit in the space its type allows.

Older versions of NumPy did not stop you: they quietly wrapped −5 round to **251**, as `uint8` sums wrapped in A8, and the "background" became a valid-looking code. Today's error is a gift.

Two fixes, depending on what you need:

- keep `uint8` and choose a fill inside its range, such as `fill=0` (and later write `nodata=0` to the file),
- or choose a type with room for negatives: `dtype="int16"` with `fill=-5`.

**Try it:** change `dtype` to `"int16"` and run the cell again.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · Landslides per district, with terrain

Join `landslides_utm` to `near` with `gpd.sjoin` (B9, inner, `predicate="within"`). Add the `elev_m` and `slope_deg` columns of `slides` to the joined table. Then, per `district`, print the number of landslides, their mean elevation and their mean slope.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
joined = gpd.sjoin(landslides_utm, near, how="inner", predicate="within")
print(len(joined))
joined["elev_m"] = slides["elev_m"]           # matched by row label, 0 to 59
joined["slope_deg"] = slides["slope_deg"]

summary = joined.groupby("district")[["elev_m", "slope_deg"]].mean()
summary["n"] = joined.groupby("district").size()
print(summary.round(1))
```

All 60 points find a district. Sindhupalchok has the most (21), then Nuwakot (20) and Kabhrepalanchok (11). The joined table keeps the landslides' row labels, and `slides` has the same labels, so pandas matches each value to the right landslide. `groupby(...).size()` counts the rows per district (B9), and pandas lines the counts up with the means by district name. Remember the points are made-up.
</details>

### Review 2 · How big is the edge effect?

Repeat the nearest-headquarters search with **all 75** headquarters: `gpd.sjoin_nearest(landslides_utm, hq, distance_col="hq_m")`. Store the distance in km in `slides["hq_all_km"]`. Count the landslides whose raster distance (`hq_km`) is more than 1 km too large, and list which headquarters outside the grid are their nearest.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
nearest_all = gpd.sjoin_nearest(landslides_utm, hq, distance_col="hq_m")
slides["hq_all_km"] = nearest_all["hq_m"] / 1000     # matched by row label
slides["nearest_hq"] = nearest_all["name"]

too_far = slides[slides["hq_km"] - slides["hq_all_km"] > 1]
print(len(too_far), "landslides")
print(too_far["nearest_hq"].value_counts())
print(round(slides["hq_km"].max(), 1), round(slides["hq_all_km"].max(), 1))
```

14 landslides are more than 1 km closer to a headquarters outside the grid: Dhunche (Rasuwa) for 10 and Bidur (Nuwakot) for 4. The largest distance drops from about 33.0 km to about 26.2 km. A distance raster is only as good as the features inside it: include features beyond the edges of your study area.
</details>

### Review 3 · Save the zone grid as a GeoTIFF

Write `zones` to `../outputs/c07_district_zones.tif` with `rasterio.open(..., "w", ...)` as in C1: one band, `dtype="uint8"`, the slope grid's CRS and transform, and `nodata=0`. Read it back and print its dtype, nodata value and the unique codes.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
profile = {"driver": "GTiff", "height": zones.shape[0], "width": zones.shape[1], "count": 1,
           "dtype": "uint8", "crs": slope.rio.crs, "transform": slope.rio.transform(), "nodata": 0}
with rasterio.open(OUT / "c07_district_zones.tif", "w", **profile) as dst:
    dst.write(zones, 1)

with rasterio.open(OUT / "c07_district_zones.tif") as src:
    back = src.read(1)
    print(src.dtypes[0], src.nodata, np.unique(back))
```

The file holds `uint8` codes 1 to 9 with `nodata` 0, which matches the `fill` used in `rasterize`. Keep the dtype, the fill and the nodata value consistent, and the file will read back exactly as you wrote it. `**profile` passes the dictionary's items as keyword arguments.
</details>

### Review 4 · Steep ground per district

Using `zones`, `valid` and `steep`, loop over the districts in `near` (use `zip` on `near["zone"]` and `near["district"]`). For each, print the km² with data, the km² steeper than 35°, and the percentage that is steep.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
for code, name in zip(near["zone"], near["district"]):
    in_zone = (zones == code) & valid
    zone_km2 = in_zone.sum() * cell_km2
    steep_km2 = (in_zone & steep).sum() * cell_km2
    print(f"{name:16} {zone_km2:7.1f} km² {steep_km2:6.1f} km² steep {steep_km2 / zone_km2 * 100:5.1f}%")
```

Sindhupalchok's covered part has the most steep ground (about 221.6 km², 22.3%). Makawanpur's small covered part has the highest share (29.8%), and Bhaktapur the lowest (1.6%). `{name:16}` pads the name to 16 characters so the columns line up. With a zone grid, any question about "how much of X inside each polygon" becomes a few NumPy masks.
</details>

## Recap

- Rasters and vectors must share a CRS before they meet. For slope and distance, work on a projected grid in metres (UTM 45N).
- `src.sample([(x, y), ...])` reads a file at many points; `da.sel(x=xr.DataArray(..., dims="point"), y=..., method="nearest")` reads a DataArray. Without `dims="point"` you get every combination instead of pairs.
- Compare values at points with the whole grid (medians, shares, frequency ratios) before drawing conclusions.
- `zonal_stats` summarises a raster per polygon; `rasterize` burns polygons into a grid with a `dtype` that holds every value and the `fill`.
- `distance_transform_edt(mask, sampling=cell_m)` gives distances in metres, but sees only features inside the grid; `features.shapes` turns a mask back into polygons.

### Words you learned

| Word | Meaning |
| --- | --- |
| `sample()` | rasterio method that reads raster values at a list of (x, y) points |
| generator | A source that hands out results one at a time as you loop over it |
| pointwise selection | Choosing one cell per (x, y) pair with `.sel`, by giving x and y a shared dimension |
| `method="nearest"` | Pick the cell whose centre is closest to the coordinate |
| background | All valid cells of a grid, used for comparison with values at points |
| frequency ratio | Share of landslides in a class divided by the share of ground in that class |
| susceptibility | How prone ground is to landslides, judged from factors such as slope |
| zonal statistics | Summaries (count, mean, max ...) of raster cells inside each polygon |
| zone | A polygon, or a group of cells, that statistics are calculated for |
| rasterize | Turn vector shapes into grid cells |
| fill | The value `rasterize` gives to cells that no shape covers |
| overflow | A value too large or too small for its data type |
| distance raster | A grid holding, in each cell, the distance to the nearest feature |
| Euclidean distance transform (EDT) | An algorithm that computes straight-line distances to the nearest feature cell |
| edge effect | Error near the edge of a study area, caused by features a short way outside it |
| polygonize | Turn patches of equal cell values into polygons |
| connectivity | Whether cells join through edges only (4) or also through corners (8) |

## Next

**C8 · Interpolation: from points to a surface:** turn 24 made-up rain gauges into a rainfall map with nearest neighbour, inverse distance weighting and kriging, and test honestly how good each map is.

<!-- pygile-nav -->
**PyGILE Course** · lesson 30 of 36 · [Course home](../README.md)  
← Previous: [C6 · Terrain analysis: slope, aspect and hillshade for landslide studies](C06_Terrain_analysis.ipynb) · Next: [C8 · Interpolation: a rainfall map from 24 gauges](C08_Interpolation.ipynb) →